## 量化FP32模型并导出单个测试图片计算过程中的所有INT8数据
1. **其中Scale参数来源于训练集采样** 

In [2]:
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

# ==========================================
# 0. 定义手动展平函数
# ==========================================
def my_flatten(x):
    """
    手动实现展平操作，严格按照 C/C++ 的行优先 (Row-Major) 逻辑。
    """
    B, C, H, W = x.shape
    flattened_memory = []
    for b in range(B):
        for c in range(C):
            for h in range(H):
                for w in range(W):
                    val = x[b, c, h, w].item() 
                    flattened_memory.append(val)
    return torch.tensor([flattened_memory], dtype=x.dtype, device=x.device)

# ==========================================
# 1. 完善后的全网定义
# ==========================================
class HardwareFriendlyCNN(torch.nn.Module):
    def __init__(self):
        super(HardwareFriendlyCNN, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = torch.nn.MaxPool2d(2, 2)
        self.conv2 = torch.nn.Conv2d(16, 32, kernel_size=3, padding=1)
        # 注意这里的 32*8*8，说明经历了两次 2x2 的池化 (32 -> 16 -> 8)
        self.fc = torch.nn.Linear(32 * 8 * 8, 10)
        self.relu = torch.nn.ReLU()
        
    def forward(self, x):
        # 这个前向传播严格定义了数据的物理流向
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) # 训练用的 flatten，也是用于展平的， -1表示由python自动推断这一维度的大小。
        x = self.fc(x)
        return x

# ==========================================
# 2. 量化核心工具函数
# ==========================================
def sym_quantize(tensor):
    """对称量化：强制 Zero Point = 0"""
    max_abs = tensor.abs().max().item()
    scale = max_abs / 127.0 if max_abs > 0 else 1.0
    q_tensor = torch.round(tensor / scale)
    q_tensor = torch.clamp(q_tensor, -127, 127).to(torch.int8)
    return q_tensor, scale

def export_c_array(tensor, name, file_handle, c_type="int8_t"):
    """导出为一维 C 数组"""
    flat_data = tensor.flatten().tolist() #将 PyTorch 底层的 C++ 张量对象，转换成纯粹的 Python 列表，用于for循环
    file_handle.write(f"const {c_type} {name}[{len(flat_data)}] = {{\n    ") #{{ 的转义机制
    for i, val in enumerate(flat_data):#enumerate 会同时返回数据的当前索引 i 和具体的数值 val
        file_handle.write(f"{val}, ")
        if (i + 1) % 16 == 0:
            file_handle.write("\n    ")
    file_handle.write("\n};\n\n")

def calc_multiplier_shift(s_in, s_w, s_out, shift=16):
    """计算硬件所需的 Multiplier，Shift 固定为 16 以简化硬件设计"""# multiplier是用于重新量化，Shift是其被放大的位数
    M = (s_in * s_w) / s_out
    multiplier = int(round(M * (1 << shift)))
    return multiplier, shift

# ==========================================
# 3. 主流程：计算 Scale，生成黄金模型
# ==========================================
def main():
    print(">>> 步骤 1/4: 加载模型与数据集...")
    model = HardwareFriendlyCNN()
    model.load_state_dict(torch.load('my_hardware_cnn_fp32_69p.pth', weights_only=True)) #防御机制,防止执行脚本
    model.eval()
    #Compose是流水线打包盒
    transform = transforms.Compose([transforms.ToTensor()])#原始图片的像素值是 0 到 255 的 uint8 整数，ToTensor() 会自动将这些整数除以 255
    
    # 【规范化升级】使用训练集作为校准集，彻底规避数据泄露！
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    # 取前 1000 张训练集图片作为校准数据足够了
    calib_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(trainset, range(1000)), batch_size=100, shuffle=False)
    
    # 加载测试集，只为了提取我们的“黄金测试向量 1”
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    
    # 【锁定目标】提取测试集 Index = 1 的图片
    TARGET_INDEX = 1
    target_img_fp32, target_label = testset[TARGET_INDEX]
    target_img_fp32 = target_img_fp32.unsqueeze(0) #维度扩充，从(3, 32, 32) 变成 (1, 3, 32, 32)
    print(f"提取黄金测试图片成功 (Index={TARGET_INDEX})！真实标签: {target_label}")

    print("\n>>> 步骤 2/4: 执行严格的 PTQ 静态校准 (基于训练集)...")
    global_max_in, global_max_out1, global_max_out2, global_max_fc = 0.0, 0.0, 0.0, 0.0

    with torch.no_grad():#使用F中的各种标准函数：精妙之处在于“拦截数据流”
        for images, _ in calib_loader:
            global_max_in = max(global_max_in, images.abs().max().item())
            out1 = F.conv2d(images, model.conv1.weight, model.conv1.bias, padding=1)
            global_max_out1 = max(global_max_out1, out1.abs().max().item())
            pool1 = F.max_pool2d(F.relu(out1), 2)
            out2 = F.conv2d(pool1, model.conv2.weight, model.conv2.bias, padding=1)
            global_max_out2 = max(global_max_out2, out2.abs().max().item())
            pool2 = F.max_pool2d(F.relu(out2), 2)
            flat = pool2.view(pool2.size(0), -1)
            fc_out = F.linear(flat, model.fc.weight, model.fc.bias)
            global_max_fc = max(global_max_fc, fc_out.abs().max().item())

    # 更加硬件友好的安全写法：如果最大值为 0（极罕见），强制设为 1.0 防止 M 爆炸
    s_in = global_max_in / 127.0 if global_max_in > 0 else 1.0
    s_out1 = global_max_out1 / 127.0 if global_max_out1 > 0 else 1.0
    s_out2 = global_max_out2 / 127.0 if global_max_out2 > 0 else 1.0
    s_out_fc = global_max_fc / 127.0 if global_max_fc > 0 else 1.0

    print("\n>>> 步骤 3/4: 固化量化参数与权重...")
    # Layer 1
    w1_q, s_w1 = sym_quantize(model.conv1.weight)
    b1_q = torch.round(model.conv1.bias / (s_in * s_w1)).to(torch.int32)
    mult1, shift1 = calc_multiplier_shift(s_in, s_w1, s_out1)
    
    # Layer 2
    w2_q, s_w2 = sym_quantize(model.conv2.weight)
    b2_q = torch.round(model.conv2.bias / (s_out1 * s_w2)).to(torch.int32)
    mult2, shift2 = calc_multiplier_shift(s_out1, s_w2, s_out2)
    
    # Layer 3 (FC)
    wfc_q, s_wfc = sym_quantize(model.fc.weight)
    bfc_q = torch.round(model.fc.bias / (s_out2 * s_wfc)).to(torch.int32)
    mult_fc, shift_fc = calc_multiplier_shift(s_out2, s_wfc, s_out_fc)

    print("\n>>> 步骤 4/4: 为靶向目标 (Index=1) 生成 Bit-Accurate 黄金输出...")
    # 1. 严格使用全局 s_in 量化输入
    target_img_q = torch.round(target_img_fp32 / s_in)
    target_img_q = torch.clamp(target_img_q, -127, 127).to(torch.int8)

    # 2. 纯 INT8 前向推演
    mac1 = F.conv2d(target_img_q.float(), w1_q.float(), bias=b1_q.float(), padding=1).to(torch.int32)
    q_conv1_out = torch.clamp((mac1 * mult1) >> shift1, -127, 127).to(torch.int8)
    q_pool1_out = F.max_pool2d(F.relu(q_conv1_out.float()), 2).to(torch.int8)

    mac2 = F.conv2d(q_pool1_out.float(), w2_q.float(), bias=b2_q.float(), padding=1).to(torch.int32)
    q_conv2_out = torch.clamp((mac2 * mult2) >> shift2, -127, 127).to(torch.int8)
    q_pool2_out = F.max_pool2d(F.relu(q_conv2_out.float()), 2).to(torch.int8)

    q_flat_in = my_flatten(q_pool2_out)
    mac_fc = F.linear(q_flat_in.float(), wfc_q.float(), bias=bfc_q.float()).to(torch.int32)
    q_fc_out = torch.clamp((mac_fc * mult_fc) >> shift_fc, -127, 127).to(torch.int8)

    predicted_label = torch.argmax(q_fc_out, dim=1).item()
    print(f"🎯 INT8 黄金模型对 Target 的预测结果: {predicted_label} (对比真实标签: {target_label})")

    # 导出到 test_vector.h ...
    with open("test_vector.h", "w") as f:
        f.write("#ifndef TEST_VECTOR_H\n#define TEST_VECTOR_H\n\n#include <stdint.h>\n\n")
        f.write("// ================= 全局统一量化参数 =================\n")
        
        # 将 s_in 存为浮点常量 (如果预处理在单片机/PicoRV32上做，会用到它；如果全由PC处理好再传进去，则仅作记录)
        f.write(f"const float GLOBAL_S_IN = {s_in};\n\n")
        
        # 导出 Layer 1 量化参数
        f.write(f"const int32_t L1_MULT = {mult1};\n")
        f.write(f"const int32_t L1_SHIFT = {shift1};\n\n")
        
        # 导出 Layer 2 量化参数
        f.write(f"const int32_t L2_MULT = {mult2};\n")
        f.write(f"const int32_t L2_SHIFT = {shift2};\n\n")
        
        # 导出 FC 层量化参数
        f.write(f"const int32_t FC_MULT = {mult_fc};\n")
        f.write(f"const int32_t FC_SHIFT = {shift_fc};\n\n")
        
        export_c_array(target_img_q, "test_input_img", f, "int8_t")
        
        # 导出 Layer 1
        f.write(f"// ================= Conv1 阶段 =================\n")
        export_c_array(w1_q, "test_conv1_weight", f, "int8_t")
        export_c_array(b1_q, "test_conv1_bias", f, "int32_t")
        f.write(f"// 黄金对照组：第一层 Conv + ReLU + Pool 后的输出结果 (大小 16x16x16=4096)\n")
        export_c_array(q_pool1_out, "expected_pool1_output", f, "int8_t")
        
        # 导出 Layer 2
        f.write(f"// ================= Conv2 阶段 =================\n")
        export_c_array(w2_q, "test_conv2_weight", f, "int8_t")
        export_c_array(b2_q, "test_conv2_bias", f, "int32_t")
        f.write(f"// 黄金对照组：第二层 Conv + ReLU + Pool 后的输出结果 (大小 32x8x8=2048)\n")
        export_c_array(q_pool2_out, "expected_pool2_output", f, "int8_t")
        
        # 导出 Layer 3 (FC)
        f.write(f"// ================= FC 阶段 =================\n")
        export_c_array(wfc_q, "test_fc_weight", f, "int8_t")
        export_c_array(bfc_q, "test_fc_bias", f, "int32_t")
        f.write(f"// 黄金对照组：全连接层最终 10 分类输出\n")
        export_c_array(q_fc_out, "expected_fc_output", f, "int8_t")

        f.write("#endif // TEST_VECTOR_H\n")
    print("静态全局量化的黄金对账单导出完毕！")

if __name__ == "__main__":
    main()

>>> 步骤 1/4: 加载模型与数据集...
Files already downloaded and verified
Files already downloaded and verified
提取黄金测试图片成功 (Index=1)！真实标签: 8

>>> 步骤 2/4: 执行严格的 PTQ 静态校准 (基于训练集)...

>>> 步骤 3/4: 固化量化参数与权重...

>>> 步骤 4/4: 为靶向目标 (Index=1) 生成 Bit-Accurate 黄金输出...
🎯 INT8 黄金模型对 Target 的预测结果: 8 (对比真实标签: 8)
静态全局量化的黄金对账单导出完毕！


将test_vector.h中图像数据从CHW格式转化为HWC格式，并且输出到test_vector_img2.h中去，排列格式为RGB

In [ ]:
import numpy as np
import re

def convert_format(input_filename, output_filename):
    # 1. 读取原始 C 语言头文件
    try:
        with open(input_filename, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"错误：未找到文件 {input_filename}，请检查路径。")
        return

    # 2. 使用正则表达式提取大括号内的所有数字
    # 匹配 const int8_t 数组声明及大括号内的内容
    match = re.search(r'const\s+int8_t\s+([a-zA-Z0-9_]+)\[\d+\]\s*=\s*\{([^}]+)\};', content)
    if not match:
        print("解析失败：未在文件中找到匹配的 C 语言数组声明格式。")
        return

    original_array_name = match.group(1)
    data_str = match.group(2)

    # 提取所有带有可选负号的整数
    numbers = [int(x) for x in re.findall(r'-?\d+', data_str)]
    
    if len(numbers) != 3072:
        print(f"警告：提取到的数据数量为 {len(numbers)}，预期为 3072！")
        return

    # 3. 使用 NumPy 进行数据重排 (CHW -> HWC)
    # 原始形状: Channel=3, Height=32, Width=32
    arr_chw = np.array(numbers, dtype=np.int8).reshape(3, 32, 32)
    
    # 维度转置：CHW(0, 1, 2) -> HWC(1, 2, 0)
    arr_hwc = np.transpose(arr_chw, (1, 2, 0))
    
    # 将转换后的多维数组展平为一维
    arr_hwc_flat = arr_hwc.flatten()

    # 4. 将转换后的数据格式化为 C 语言头文件风格
    out_lines = []
    # 为避免重名，给新数组名称加上 _hwc 后缀
    new_array_name = f"{original_array_name}_hwc"
    out_lines.append(f"const int8_t {new_array_name}[3072] = {{")
    
    # 每行输出 16 个数字，保持代码整洁
    for i in range(0, len(arr_hwc_flat), 16):
        chunk = arr_hwc_flat[i:i+16]
        # 将数字转换为字符串并用逗号空格拼接
        chunk_str = ", ".join(map(str, chunk))
        out_lines.append(f"    {chunk_str},")
        
    out_lines.append("};")

    # 5. 写入目标文件
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("\n".join(out_lines) + "\n")
        
    print(f"转换成功！CHW 格式已转为 HWC。")
    print(f"已生成包含 {len(arr_hwc_flat)} 个元素的新文件：{output_filename}")

if __name__ == "__main__":
    # 执行转换
    convert_format("test_vector.h", "test_vector_img2.h")

转换成功！CHW 格式已转为 HWC。
已生成包含 3072 个元素的新文件：test_vector_2.h


将test_vector.h中图像数据从CHW格式转化为HWC格式，并且覆写到test_vector_img2.h中去，排列格式为BGR

In [2]:
import numpy as np
import re

def convert_format_to_bgr(input_filename, output_filename):
    # 1. 读取原始 C 语言头文件
    try:
        with open(input_filename, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"错误：未找到文件 {input_filename}，请检查路径。")
        return

    # 2. 使用正则表达式提取大括号内的所有数字
    match = re.search(r'const\s+int8_t\s+([a-zA-Z0-9_]+)\[\d+\]\s*=\s*\{([^}]+)\};', content)
    if not match:
        print("解析失败：未在文件中找到匹配的 C 语言数组声明格式。")
        return

    original_array_name = match.group(1)
    data_str = match.group(2)

    # 提取所有带有可选负号的整数
    numbers = [int(x) for x in re.findall(r'-?\d+', data_str)]
    
    if len(numbers) != 3072:
        print(f"警告：提取到的数据数量为 {len(numbers)}，预期为 3072！")
        return

    # 3. 使用 NumPy 进行数据重排 (CHW_RGB -> HWC_BGR)
    # 原始形状: Channel=3 (RGB), Height=32, Width=32
    arr_chw_rgb = np.array(numbers, dtype=np.int8).reshape(3, 32, 32)
    
    # 【核心修改点 1】：逆序通道。将第 0 维 (Channel 维) 从 R-G-B (0, 1, 2) 翻转为 B-G-R (2, 1, 0)
    arr_chw_bgr = arr_chw_rgb[::-1, :, :]
    
    # 【核心修改点 2】：维度转置。CHW(0, 1, 2) -> HWC(1, 2, 0)
    arr_hwc_bgr = np.transpose(arr_chw_bgr, (1, 2, 0))
    
    # 将转换后的多维数组展平为一维
    arr_hwc_flat = arr_hwc_bgr.flatten()

    # 4. 将转换后的数据格式化为 C 语言头文件风格
    out_lines = []
    # 为避免重名，给新数组名称加上 _hwc_bgr 后缀
    new_array_name = f"{original_array_name}_hwc_bgr"
    out_lines.append(f"const int8_t {new_array_name}[3072] = {{")
    
    # 每行输出 16 个数字，保持代码整洁
    for i in range(0, len(arr_hwc_flat), 16):
        chunk = arr_hwc_flat[i:i+16]
        # 将数字转换为字符串并用逗号空格拼接
        chunk_str = ", ".join(map(str, chunk))
        out_lines.append(f"    {chunk_str},")
        
    out_lines.append("};")

    # 5. 写入目标文件
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("\n".join(out_lines) + "\n")
        
    print(f"转换成功！CHW (RGB) 格式已转为 HWC (BGR)。")
    print(f"已生成包含 {len(arr_hwc_flat)} 个元素的新文件：{output_filename}")

if __name__ == "__main__":
    # 执行转换
    convert_format_to_bgr("test_vector.h", "test_vector_img2.h")

转换成功！CHW (RGB) 格式已转为 HWC (BGR)。
已生成包含 3072 个元素的新文件：test_vector_2.h


将test_vector_img2.h中的test_input_img_hwc_bgr每3个数重新拼接为一个32bit数,输出到test_vector_img3.h中

In [3]:
import re

def pack_bgr_to_32bit(input_filename, output_filename):
    # 1. 读取上一步生成的 HWC 格式头文件
    try:
        with open(input_filename, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"错误：未找到文件 {input_filename}，请先运行上一个脚本。")
        return

    # 2. 使用正则表达式提取数据
    match = re.search(r'const\s+int8_t\s+([a-zA-Z0-9_]+)\[\d+\]\s*=\s*\{([^}]+)\};', content)
    if not match:
        print("解析失败：未在文件中找到匹配的 C 语言数组声明格式。")
        return

    original_array_name = match.group(1)
    data_str = match.group(2)

    numbers = [int(x) for x in re.findall(r'-?\d+', data_str)]
    
    if len(numbers) != 3072:
        print(f"警告：提取到的数据数量为 {len(numbers)}，预期为 3072！")
        return

    # 3. 核心拼接逻辑：每 3 个 8-bit 数据拼成一个 32-bit 数据
    packed_numbers = []
    for i in range(0, len(numbers), 3):
        b = numbers[i]
        g = numbers[i+1]
        r = numbers[i+2]
        
        # 必须使用 & 0xFF 清除 Python 负数的符号位扩展，模拟真实的物理线拼接
        packed_val = ((b & 0xFF) << 16) | ((g & 0xFF) << 8) | (r & 0xFF)
        packed_numbers.append(packed_val)

    # 4. 格式化输出为 C 语言头文件
    out_lines = []
    # 更改数组名为 packed 并且类型改为 uint32_t
    new_array_name = original_array_name.replace("hwc_bgr", "packed32")
    if new_array_name == original_array_name:
        new_array_name += "_packed32"
        
    out_lines.append(f"// 该数组由 3 个 8-bit 数据拼成一个 32-bit 字: [23:16]=B, [15:8]=G, [7:0]=R")
    out_lines.append(f"const uint32_t {new_array_name}[1024] = {{")
    
    # 每行输出 8 个 32-bit 数据，使用十六进制打印方便硬件调试波形比对
    for i in range(0, len(packed_numbers), 8):
        chunk = packed_numbers[i:i+8]
        # 格式化为 0x00XXXXXX 的标准 6/8 位十六进制
        chunk_str = ", ".join([f"0x{x:06X}" for x in chunk])
        out_lines.append(f"    {chunk_str},")
        
    out_lines.append("};")

    # 5. 写入目标文件
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("\n".join(out_lines) + "\n")
        
    print(f"数据打包成功！3072 个 8-bit 数据已合并为 1024 个 32-bit 字。")
    print(f"已生成新文件：{output_filename}")

if __name__ == "__main__":
    # 执行转换
    pack_bgr_to_32bit("test_vector_img2.h", "test_vector_img3.h")

数据打包成功！3072 个 8-bit 数据已合并为 1024 个 32-bit 字。
已生成新文件：test_vector_3.h


将上述文件中输入激活值改成便于给SRAM赋值的形式

In [7]:
import re

def format_image_for_readmemh(input_filename, output_filename, target_var="test_input_img_packed32"):
    # ==========================================
    # 1. 读取原始数据文件
    # ==========================================
    try:
        with open(input_filename, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"错误：未找到文件 {input_filename}，请检查路径。")
        return

    # ==========================================
    # 2. 精确正则匹配 (精准狙击特定的变量名)
    # ==========================================
    # 匹配目标数组的大括号内容
    pattern = rf'const\s+uint32_t\s+({target_var})\[\d+\]\s*=\s*\{{([^}}]+)\}};'
    
    match = re.search(pattern, content)
    if not match:
        print(f"❌ 解析失败：未在文件中找到名为 '{target_var}' 的 uint32_t 数组！")
        return

    extracted_name = match.group(1)
    data_str = match.group(2)

    # ==========================================
    # 3. 提取十六进制数据并转换
    # ==========================================
    # 直接提取所有形如 0x757575 的十六进制字符串（忽略大小写）
    hex_strings = re.findall(r'0x[0-9a-fA-F]+', data_str)
    
    if len(hex_strings) != 1024:
        print(f"⚠️ 警告：提取到的数据数量为 {len(hex_strings)}，预期为 1024！")
        # 不直接 return，允许强制输出以便排查

    # ==========================================
    # 4. 核心转换逻辑 (补充前导 0)
    # ==========================================
    out_lines = []
    # 写入 Verilog readmemh 能够识别的起始地址标记
    out_lines.append("@00004000    // Image 区域 (Word Index = 0x10000 / 4 = 0x4000)")

    for hs in hex_strings:
        # 将 "0x757575" 转为整数，然后再格式化为 8 位大写十六进制字符串 (自动补齐前导 0)
        val = int(hs, 16)
        out_lines.append(f"{val:08X}")

    # ==========================================
    # 5. 写入目标文件
    # ==========================================
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("\n".join(out_lines) + "\n")
        
    print(f"✅ 成功从数组 [{extracted_name}] 提取图像特征数据！")
    print(f"📦 共生成 {len(hex_strings)} 行 32-bit 格式化数据。")
    print(f"💾 数据已成功保存至：{output_filename}")

if __name__ == "__main__":
    # 执行格式化提取
    # 假设你的常量数组存在 test_vector_3.h 中
    format_image_for_readmemh("test_vector_img3.h", "image_readmemh.txt", target_var="test_input_img_packed32")

✅ 成功从数组 [test_input_img_packed32] 提取图像特征数据！
📦 共生成 1024 行 32-bit 格式化数据。
💾 数据已成功保存至：image_readmemh.txt


## 将权重数据打包成便于给SRAM赋值的格式

In [3]:
import re

def pack_weights_for_npu(input_filename, output_filename):
    # ==========================================
    # 1. 读取原始 C 语言头文件
    # ==========================================
    try:
        with open(input_filename, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"错误：未找到文件 {input_filename}，请检查路径。")
        return

    # ==========================================
    # 2. 使用正则表达式精准提取大括号内的数字
    # ==========================================
    match = re.search(r'const\s+int8_t\s+([a-zA-Z0-9_]+)\[\d+\]\s*=\s*\{([^}]+)\};', content)
    if not match:
        print("解析失败：未在文件中找到匹配的 C 语言数组声明格式。")
        return

    original_array_name = match.group(1)
    data_str = match.group(2)

    # 提取所有带有可选负号的整数
    numbers = [int(x) for x in re.findall(r'-?\d+', data_str)]
    
    if len(numbers) != 432:
        print(f"警告：提取到的数据数量为 {len(numbers)}，预期为 432！请检查输入文件。")
        return

    # ==========================================
    # 3. 核心打包逻辑 (为 4x4 脉动阵列重组)
    # ==========================================
    out_lines = []
    # 写入 Verilog readmemh 能够识别的地址标记
    out_lines.append("@00000400    // Weight 区域 (Word Index = 0x1000 / 4 = 0x400)")

    # 总共分为 4 大组 (对应阵列 4 次时分复用计算)
    for batch_idx in range(4):
        # 每一大组包含 4 个输出通道 (对应 4 列 PE)
        for oc_idx in range(4):
            # 当前 OC (27个数) 的全局起始偏移量
            offset = (batch_idx * 4 + oc_idx) * 27
            
            # 遍历 3x3=9 个空间位置
            for pos in range(9):
                # 获取三个输入通道对应的索引位置
                idx_ic2 = offset + 18 + pos
                idx_ic1 = offset + 9 + pos
                idx_ic0 = offset + pos
                
                # 提取数据，使用 & 0xFF 截断 Python 负数的无限符号扩展
                v2 = numbers[idx_ic2] & 0xFF
                v1 = numbers[idx_ic1] & 0xFF
                v0 = numbers[idx_ic0] & 0xFF
                
                # 拼接为 32-bit：[31:24]补0, [23:16]=IC2, [15:8]=IC1, [7:0]=IC0
                packed_val = (v2 << 16) | (v1 << 8) | v0
                
                # 格式化输出为 8 位全大写十六进制字符串（例如: 00C3C4B4）
                out_lines.append(f"{packed_val:08X}")

    # ==========================================
    # 4. 写入目标文件
    # ==========================================
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("\n".join(out_lines) + "\n")
        
    print(f"✅ 成功从数组 [{original_array_name}] 提取特征数据！")
    print(f"📦 打包完成！共生成 {len(out_lines) - 1} 行 32-bit 权重数据。")
    print(f"💾 数据已成功保存至：{output_filename}")

if __name__ == "__main__":
    # 执行打包
    pack_weights_for_npu("test_vector.h", "weights_readmemh.txt")

✅ 成功从数组 [test_conv1_weight] 提取特征数据！
📦 打包完成！共生成 144 行 32-bit 权重数据。
💾 数据已成功保存至：weights_readmemh.txt


对上一步生成的权重数据进行重排，以实现输出通道数据交织排列

In [4]:
import os

def interleave_weights_for_fsm(input_filename, output_filename):
    # ==========================================
    # 1. 读取上一步生成的 txt 文件
    # ==========================================
    try:
        with open(input_filename, 'r', encoding='utf-8') as f:
            lines = f.read().splitlines()
    except FileNotFoundError:
        print(f"错误：找不到文件 {input_filename}，请先运行上一个脚本。")
        return

    header = ""
    data_hex = []

    # ==========================================
    # 2. 解析文件：分离 Header 和 核心数据
    # ==========================================
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith('@'):
            header = line
        else:
            # 提取十六进制数据（防止后面有注释）
            hex_val = line.split('//')[0].strip()
            if hex_val:
                data_hex.append(hex_val)

    if len(data_hex) != 144:
        print(f"警告：读取到的数据数量为 {len(data_hex)}，预期为 144！")
        return

    # ==========================================
    # 3. 核心交织逻辑 (Transpose Data Layout)
    # ==========================================
    interleaved_data = []
    
    # 总共分为 4 大组 (每组 36 个数据，对应 4 次 NPU 时分复用)
    for batch_idx in range(4):
        # 提取当前这 36 个数据
        macro_group = data_hex[batch_idx * 36 : (batch_idx + 1) * 36]
        
        # 在这 36 个数据中，有 4 个微组 (对应 4 列 PE)，每组 9 个空间位置
        # 我们需要把 4x9 的排列转置为 9x4 的交织排列
        for pos in range(9):
            for oc_idx in range(4):
                # oc_idx * 9 是每个微组的起始偏移量 (0, 9, 18, 27)
                original_idx = oc_idx * 9 + pos
                interleaved_data.append(macro_group[original_idx])

    # ==========================================
    # 4. 写入交织后的目标文件
    # ==========================================
    with open(output_filename, 'w', encoding='utf-8') as f:
        # 还原地址标记头
        if header:
            f.write(header + "\n")
            
        # 写入交织后的数据，这里我加了一个小小的美化：
        # 每写入 4 个数据空一行，方便你在 VCS 里肉眼核对一个 FSM 节拍抓取的数据包！
        for i in range(0, len(interleaved_data), 4):
            chunk = interleaved_data[i:i+4]
            f.write("\n".join(chunk) + "\n")
            # f.write("\n") # 如果你觉得空行影响 Verilog 读取，可以把这句注释掉

    print(f"✅ 交织重排完成！144 个字已按照 FSM 节拍完美对齐。")
    print(f"💾 最终用于仿真的数据已保存至：{output_filename}")

if __name__ == "__main__":
    # 执行交织
    interleave_weights_for_fsm("weights_readmemh.txt", "weights_interleaved.txt")

✅ 交织重排完成！144 个字已按照 FSM 节拍完美对齐。
💾 最终用于仿真的数据已保存至：weights_interleaved.txt


转化偏置Bias数据为易于给SRAM赋值的格式

In [6]:
import re

def extract_specific_bias(input_filename, output_filename, target_var="test_conv1_bias"):
    # ==========================================
    # 1. 读取原始数据文件
    # ==========================================
    try:
        with open(input_filename, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"错误：未找到文件 {input_filename}，请检查路径。")
        return

    # ==========================================
    # 2. 精确正则匹配 (精准狙击特定的变量名)
    # ==========================================
    # 使用 f-string 动态构建正则表达式，将 target_var 锁定在语法中
    # 注意：这里的 \{{ 和 \}} 是为了在 f-string 中转义大括号
    pattern = rf'const\s+int32_t\s+({target_var})\[\d+\]\s*=\s*\{{([^}}]+)\}};'
    
    match = re.search(pattern, content)
    if not match:
        print(f"❌ 解析失败：未在文件中找到名为 '{target_var}' 的 int32_t 数组！")
        print("请检查变量名拼写是否正确，或数据类型是否为 int32_t。")
        return

    extracted_name = match.group(1)
    data_str = match.group(2)

    # 提取所有带有可选负号的整数
    numbers = [int(x) for x in re.findall(r'-?\d+', data_str)]
    
    if len(numbers) != 16:
        print(f"⚠️ 警告：提取到的数据数量为 {len(numbers)}，预期为 16！")
        return

    # ==========================================
    # 3. 核心转换逻辑 (32-bit 补码及格式化)
    # ==========================================
    out_lines = []
    # 写入 Verilog readmemh 能够识别的地址标记
    out_lines.append("@00000000    // Bias 区域 (Word Index = 0x0 / 4 = 0)")

    for val in numbers:
        # Python 中的整数是无限精度的，负数前面有无限个1。
        # 使用 & 0xFFFFFFFF 将其强制截断为标准的 32-bit 无符号十六进制补码形态。
        hex_val = val & 0xFFFFFFFF
        
        # 格式化输出：08X 保证输出为 8 位大写十六进制，后跟原数值的注释
        out_lines.append(f"{hex_val:08X}     // {val}")

    # ==========================================
    # 4. 写入目标文件
    # ==========================================
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("\n".join(out_lines) + "\n")
        
    print(f"✅ 成功从数组 [{extracted_name}] 精准提取 Bias 数据！")
    print(f"📦 共生成 {len(numbers)} 行 32-bit 格式化数据。")
    print(f"💾 数据已成功保存至：{output_filename}")

if __name__ == "__main__":
    # 执行提取（你可以随时将 "test_conv1_bias" 替换为 "test_conv2_bias"）
    # 假设你的所有常量数组都存在 test_vectors.h 中
    extract_specific_bias("test_vector.h", "bias_readmemh.txt", target_var="test_conv1_bias")

✅ 成功从数组 [test_conv1_bias] 精准提取 Bias 数据！
📦 共生成 16 行 32-bit 格式化数据。
💾 数据已成功保存至：bias_readmemh.txt


##将data.hex转化为能够给picorv32+npu联合仿真的头文件test_vector_2.h

In [1]:
import os

# ==========================================
# 1. 配置路径与参数
# ==========================================
INPUT_HEX_FILE = "./sram_data/data.hex"        # 你原来的 Verilog hex 文件
OUTPUT_HEADER_FILE = "test_vector_3.h" # 要生成的 C 语言头文件

# ==========================================
# 2. 核心解析与转换函数
# ==========================================
def convert_hex_to_header(in_file, out_file):
    bias_data = []
    weight_data = []
    image_data = []
    
    current_word_addr = 0
    
    # 2.1 解析 hex 文件
    if not os.path.exists(in_file):
        print(f"[Error] 找不到输入文件: {in_file}")
        return
        
    with open(in_file, "r") as f:
        for line in f:
            # 去除注释 (//) 和两端的空格/换行
            line = line.split("//")[0].strip()
            if not line:
                continue
            
            # 判断是否是地址跳转符 @
            if line.startswith("@"):
                current_word_addr = int(line[1:], 16)
            else:
                # 给数据加上 C 语言的十六进制前缀 0x
                c_hex_val = f"0x{line.lower()}"
                
                # 根据地址（Word Index）归类数据
                if current_word_addr < 0x400:          # @00000000 开始的是 Bias
                    bias_data.append(c_hex_val)
                elif current_word_addr < 0x4000:       # @00000400 开始的是 Weight
                    weight_data.append(c_hex_val)
                else:                                  # @00004000 开始的是 Image
                    image_data.append(c_hex_val)
                
                # 地址自增 1 (Word)
                current_word_addr += 1

    # 2.2 写入 C 头文件
    with open(out_file, "w") as f:
        f.write("// =========================================================================\n")
        f.write("// Auto-generated Header File for PicoRV32 + NPU SoC\n")
        f.write("// =========================================================================\n")
        f.write("#ifndef TEST_VECTOR_H\n")
        f.write("#define TEST_VECTOR_H\n\n")
        f.write("#include <stdint.h>\n\n")
        
        # 辅助写入函数，控制每行排版
        def write_c_array(array_name, data_list):
            # 为了防止 C 编译器报符号溢出警告，用 uint32_t 存储 16 进制常量是最安全的
            f.write(f"const uint32_t {array_name}[{len(data_list)}] = {{\n")
            for i in range(0, len(data_list), 8):
                # 每次取 8 个数据为一行
                chunk = data_list[i:i+8]
                f.write("    " + ", ".join(chunk) + ",\n")
            f.write("};\n\n")
            print(f"[Success] {array_name} 生成完毕，共 {len(data_list)} 个 32-bit Words.")

        write_c_array("test_conv1_bias", bias_data)
        write_c_array("test_conv1_weight_packed32", weight_data)
        write_c_array("test_input_img_packed32", image_data)
        
        # 预留一些我们在 C 代码中需要用到的宏参数
        f.write("// --- Quantization Parameters (示例, 请替换为真实值) ---\n")
        f.write("#define L1_MULT  104\n")
        f.write("#define L1_SHIFT 16\n\n")
        
        f.write("#endif // TEST_VECTOR_H\n")

    print(f"\n[Done] 完美打包！请查看生成的文件: {out_file}")

# 执行脚本
if __name__ == "__main__":
    convert_hex_to_header(INPUT_HEX_FILE, OUTPUT_HEADER_FILE)

[Success] test_conv1_bias 生成完毕，共 16 个 32-bit Words.
[Success] test_conv1_weight_packed32 生成完毕，共 144 个 32-bit Words.
[Success] test_input_img_packed32 生成完毕，共 1024 个 32-bit Words.

[Done] 完美打包！请查看生成的文件: test_vector_3.h
